# Squeezing Transformers onto Edge Hardware 🧠📱
**The Memory Wall**: When running LLMs on edge devices, we don't run out of compute; we run out of memory bandwidth. The biggest offender during autoregressive generation is the Key-Value (KV) cache. 

Let's look at the physical memory allocations for a sequence length of 4096 tokens.

In [3]:
import jax
import jax.numpy as jnp

# Hardware & Model Constraints
batch_size = 1
max_seq_len = 128000 # 4096  # Pushing the context window
head_dim = 64
mha_heads = 16      # Standard Multi-Head Attention

print(f"Tracking memory for: Batch Size {batch_size}, Context {max_seq_len}, Head Dim {head_dim}")

Tracking memory for: Batch Size 1, Context 128000, Head Dim 64


## 1. Vanilla Multi-Head Attention (MHA)
In standard attention, we must store both the Keys and the Values for every single head. Let's allocate that memory and see the cost per layer.

In [4]:
def get_mha_cache_size(num_layers=1):
    # Two distinct tensors required
    k_cache = jnp.zeros((batch_size, max_seq_len, mha_heads, head_dim), dtype=jnp.bfloat16)
    v_cache = jnp.zeros((batch_size, max_seq_len, mha_heads, head_dim), dtype=jnp.bfloat16)
    
    total_bytes = (k_cache.nbytes + v_cache.nbytes) * num_layers
    return total_bytes / (1024 ** 2)  # MB

mha_1L = get_mha_cache_size(1)
print(f"🔴 Vanilla MHA Cache (1 Layer): {mha_1L:.2f} MB")

🔴 Vanilla MHA Cache (1 Layer): 500.00 MB


## 2. Group Tied Attention (GTA)
To fix this, we do two things:
1. **Group the heads** (e.g., from 16 down to 8).
2. **Tie the tensors** (Keys and Values become a single matrix $A$).

Let's see what happens to our physical memory footprint.

In [5]:
gta_groups = 8 

def get_gta_cache_size(num_layers=1):
    # ONE unified tensor acts as both Key and Value
    a_cache = jnp.zeros((batch_size, max_seq_len, gta_groups, head_dim), dtype=jnp.bfloat16)
    
    total_bytes = a_cache.nbytes * num_layers
    return total_bytes / (1024 ** 2)

gta_1L = get_gta_cache_size(1)
print(f"🟢 GTA Cache (1 Layer): {gta_1L:.2f} MB")
print(f"🔥 Reduction: {mha_1L / gta_1L:.1f}x smaller!")

🟢 GTA Cache (1 Layer): 125.00 MB
🔥 Reduction: 4.0x smaller!


## 3. Depth Reinvestment vs. Truncation
Because we just shrank our memory footprint by 4x, we don't have to brutally truncate our models to fit on a mobile GPU. We can reinvest that saved memory into **depth** (adding more layers for better reasoning). 

Let's run the Iso-KV Cache Showdown.

In [4]:
# Reinvesting memory into depth
mha_6L = get_mha_cache_size(num_layers=6)
gta_24L = get_gta_cache_size(num_layers=24)
mha_24L = get_mha_cache_size(num_layers=24)

print("--- The Iso-KV Cache Showdown ---")
print(f"🔴 MHA Cache (Truncated 6 Layers): {mha_6L:.2f} MB")
print(f"🟢 GTA Cache (Deep 24 Layers):     {gta_24L:.2f} MB")

print("\n--- The Out-Of-Memory Zone ---")
print(f"💀 MHA Cache (Full 24 Layers):     {mha_24L:.2f} MB")

--- The Iso-KV Cache Showdown ---
🔴 MHA Cache (Truncated 6 Layers): 96.00 MB
🟢 GTA Cache (Deep 24 Layers):     96.00 MB

--- The Out-Of-Memory Zone ---
💀 MHA Cache (Full 24 Layers):     384.00 MB


## 4. The Positional Catch: Why not RoPE?
If we use a single tied tensor $A$, we cannot use standard Rotary Positional Embeddings (RoPE). RoPE physically rotates the vector based on its position. If you rotate a tensor to act as a Key, you destroy its geometric meaning as a Value. 

**The Solution: ALiBi (Attention with Linear Biases).**
ALiBi does not touch the KV Cache. It applies a static penalty directly to the attention logits. 

$$Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d}} + m \cdot \Delta\right)V$$

Let's prove that applying ALiBi requires **zero additional cache memory**.

In [6]:
# Simulating the Attention Logits (Q * K^T)
# Shape: (batch, heads, seq_len, seq_len)
attention_logits = jnp.zeros((batch_size, gta_groups, max_seq_len, max_seq_len), dtype=jnp.bfloat16)

# ALiBi is just a static mask added during the forward pass.
# It is computed on the fly, NOT stored in the autoregressive cache.
alibi_bias = jnp.ones((1, gta_groups, 1, max_seq_len), dtype=jnp.bfloat16) 

# The addition happens strictly in compute (SRAM), never hitting the VRAM cache capacity.
final_logits = attention_logits + alibi_bias

print(f"Size of Attention Logits: {attention_logits.nbytes / (1024**2):.2f} MB")
print(f"Size of ALiBi Bias Mask:  {alibi_bias.nbytes / (1024**2):.2f} MB")
print("\nBecause ALiBi is a broadcasted computation, it keeps our unified GTA cache mathematically clean and our memory footprint untouched.")

E0815 06:42:05.416507   20241 gpu_hlo_schedule.cc:971] The byte size of input/output arguments (262144000002) exceeds the base limit (6409863168). This indicates an error in the calculation!


Size of Attention Logits: 250000.00 MB
Size of ALiBi Bias Mask:  1.95 MB

Because ALiBi is a broadcasted computation, it keeps our unified GTA cache mathematically clean and our memory footprint untouched.


W0815 06:42:15.454728   20241 bfc_allocator.cc:514] Allocator (GPU_0_bfc) ran out of memory trying to allocate 244.14GiB (rounded to 262144000000)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
W0815 06:42:15.455044   20241 bfc_allocator.cc:525] *___________________________________________________________________________________________________
E0815 06:42:15.504148   20241 gpu_hlo_schedule.cc:971] The byte size of input/output arguments (524290048000) exceeds the base limit (6409863168). This indicates an error in the calculation!


## 5. The Zero-Copy Forward Pass
Standard attention functions expect separate $K$ and $V$ tensors. If we just pass our unified $A$ tensor into them, the compiler might duplicate it in memory, ruining our 4x compression. 

To prevent this, we use `jnp.broadcast_to`. This creates a **view** of the tensor for the math operations without allocating a single extra byte of VRAM.

In [6]:
a_cache = jnp.zeros((batch_size, max_seq_len, gta_groups, head_dim), dtype=jnp.bfloat16)
total_bytes = a_cache.nbytes
print(f"🟢 GTA Cache (1 Layer): {a_cache.nbytes / (1024 ** 2):.2f} MB")

# Create a view of the 'A' tensor to act as Keys and Values
k_view = jnp.broadcast_to(a_cache, (batch_size, max_seq_len, gta_groups, head_dim))
v_view = jnp.broadcast_to(a_cache, (batch_size, max_seq_len, gta_groups, head_dim))

print("Memory allocated for K view:", k_view.nbytes / (1024**2), "MB")
print("Notice how this didn't spike our VRAM? It's just a pointer reference!")

🟢 GTA Cache (1 Layer): 4.00 MB
Memory allocated for K view: 4.0 MB
Notice how this didn't spike our VRAM? It's just a pointer reference!
